# Experiment A: sample at T_melt/1.141 and compare with TURN

This notebook runs `src/experiment_a.py` from the Token Thermodynamics repository. The protocol, the decision rule and the budget are in `EXPERIMENT_A.md`. Read that first.

Settings to check before running. Turn on a GPU accelerator (T4 x2) and turn on internet access. For the gated Llama models, accept the licence on each model's Hugging Face page and add a Kaggle secret named `HF_TOKEN`.

In [ ]:
MODEL = "meta-llama/Llama-3.2-1B-Instruct"   # or meta-llama/Llama-3.2-3B-Instruct
TASK = "math"                                # "mbpp" executes model-written code
K = 32                                       # samples per question per temperature

# PART chooses how much to run. Each part is one "Save Version, Save & Run All".
#   "smoke"  8 problems, 4 samples, about 15 minutes, checks that everything works
#   "ratio"  only T_melt and TURN's own temperature, no accuracy grid, well under an hour
#   "A"      the accuracy grid from T = 0.1 to 0.8, about 6 hours on the 1B model
#   "B1"     T = 0.9 to 1.2, about 5 hours
#   "B2"     T = 1.3 and 1.4 plus the two T_melt settings, about 8 hours
# A, B1 and B2 together are the full run. Hot temperatures produce text that never
# stops, so every sample runs to the 1024-token cap, which is why B is split in two
# to fit Kaggle's 12 hour limit. Download results.zip from each and bring them back.
PART = "A"

GRID_A = "T0.1,T0.2,T0.3,T0.4,T0.5,T0.6,T0.7,T0.8"
GRID_B1 = "T0.9,T1.0,T1.1,T1.2"
GRID_B2 = "T1.3,T1.4,melt_task,melt_question"
OUT = f"/kaggle/working/results_{PART}"
EXTRA = {"smoke": "--n-problems 8 --k 4 --turn-samples 8 --max-new-tokens 256",
         "ratio": "--phases melt,turn",
         "A": f"--only {GRID_A}",
         "B1": f"--only {GRID_B1}",
         "B2": f"--only {GRID_B2}"}[PART]

In [ ]:
!git clone -q https://github.com/pragyaangaur/Token-Thermodynamics.git
# TURN supplies the MATH split, the four-shot prompt, the answer parser and the grader.
# Pinned to the commit the harness was written against.
!git clone -q https://github.com/StigLidu/TURN.git && git -C TURN checkout -q 64b42e22762a05f4a5ef999c868c61a83c7adb2f
!pip install -q vllm sympy pylatexenc

In [ ]:
import os, glob, shutil
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN secret, gated models will fail:", e)

# Resuming. If an earlier version of this notebook's output is attached as an input,
# copy its checkpoints so finished phases and temperatures are skipped.
for prev in glob.glob(f"/kaggle/input/*/results_{PART}"):
    shutil.copytree(prev, OUT, dirs_exist_ok=True)
    print("resumed from", prev)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import vllm; print("vllm", vllm.__version__)

In [ ]:
%cd /kaggle/working/Token-Thermodynamics
# Two separate processes. The float32 pass frees its GPU memory when its process ends,
# so vLLM starts with the whole GPU. The second command reuses the saved melt phase.
!python src/experiment_a.py --model {MODEL} --task {TASK} --k {K} --turn-dir ../TURN \
    --out {OUT} --phases melt {"--n-problems 8" if PART == "smoke" else ""}
!python src/experiment_a.py --model {MODEL} --task {TASK} --k {K} --turn-dir ../TURN \
    --out {OUT} --dtype-gen half --tp 1 {EXTRA}

If vLLM fails to start on the T4, the likely cause is that its newest release no longer supports that GPU. Try `!pip install -q "vllm<0.10"`, restart the kernel and rerun. The melt phase is already saved and will not be repeated. The summary records which kind of log probabilities vLLM returned, so the entropy curve stays comparable to TURN's.

In [ ]:
!cat {OUT}/*/*/summary.json 2>/dev/null || echo "no summary in a partial run, that is expected"
!cd /kaggle/working && zip -qr results.zip $(basename {OUT}) && ls -la results.zip